In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
pd.options.display.float_format = '{:,.1f}'.format

In [ ]:
logdir = "logs"
ignore = ["10percent", "100percent", "symmetric", "_e_optimized", "_e_split"]
ignore += ["GitPython20002010_2layers", "GitPython20002010_3layers", "GitPython20002010_4layers", "populationcountriesremoved", "populationlargean200"]
ignore += ["tree_610_649", "tree_90_99_split", "tree_27Dec", "tree_185_194", "tree_94_98"]
filerename = {"GitPython20002010": "GitPython", "GitGudhi2017": "GitGudhi", "population": "Europe", "tree_500_1400_w": "Cylinder2D", "tree_50_150_w_split": "Cylinder3D", "tree_0_742_w": "Storm", "tree_50_199_w_split": "Tangora", "GitInviwoAnimation_to2019": "GitInviwo"}
dfdict = defaultdict(list)
for file in os.listdir(logdir):
    if (any(i in file for i in ignore)): continue
    if file.endswith(".log"):
        name, rest = file.split("_optimized_")        
        with open(os.path.join(logdir, file), "r") as f:
            s = f.read()
            if "Time wiggle minimization:" not in s or "Time crossing minimization:" not in s: continue
            for line in s.splitlines():
                if line.startswith("Number of nodes: "):
                    dfdict["nodes"].append(int(line.split(": ")[1]))
                if line.startswith("Number of hierarchical edges: "):
                    dfdict["hedges"].append(int(line.split(": ")[1]))
                if line.startswith("Number of temporal edges: "):
                    dfdict["tedges"].append(int(line.split(": ")[1]))
                if line.startswith("Number of time steps: "):
                    dfdict["timesteps"].append(int(line.split(": ")[1]))
                if line.startswith("Number of levels: "):
                    dfdict["levels"].append(int(line.split(": ")[1]))
                if line.startswith("Time wiggle minimization: "):
                    dfdict["twiggle"].append(float(line.split(": ")[1].removesuffix("ms")))
                if line.startswith("Time crossing minimization: "):
                    dfdict["tcross"].append(float(line.split(": ")[1].removesuffix("ms")))
                if line.startswith("Number of crossings: "):
                    dfdict["crossings"].append(int(line.split(": ")[1]))
        if name in filerename:
            name = filerename[name]
        if "unweighted" in file:
            dfdict["weighted"].append(False)
        else:
            dfdict["weighted"].append(True)
        if "Median" in file:
            dfdict["algorithm"].append("Median")
        else:
            dfdict["algorithm"].append("ILP")
        dfdict["file"].append(name)   

df = pd.DataFrame(dfdict)
df.describe()

,nodes,hedges,tedges,timesteps,levels,tcross,crossings,twiggle
count,44.0,44.0,44.0,44.0,44.0,44.0,44.0,44.0
mean,"13,444.7","13,243.5","12,735.5",201.1,6.5,"21,000.3","21,521.0","17,907.7"
std,"33,625.8","33,629.5","31,658.3",301.9,2.3,"90,922.5","97,262.1","81,745.9"
min,132.0,127.0,102.0,5.0,3.0,1.0,0.0,6.0
25%,192.0,184.0,145.0,9.0,4.0,13.5,0.0,11.0
50%,"1,288.0","1,137.0","1,134.0",80.5,6.0,82.5,3.5,77.5
75%,"8,446.0","7,702.0","7,901.0",151.0,9.0,877.0,49.0,710.5
max,"147,843.0","147,800.0","138,841.0",902.0,10.0,"450,304.0","462,031.0","524,864.0"


In [7]:
print(df["file"].unique())

['ViscousFingers' 'Storm' 'Cartilage' 'Cylinder3D[50,150]'
 'DigestivePhysiology' 'GitPython' 'Europe' 'Tangora[40,199]'
 'Cylinder2D[500,1400]' 'UrinaryTract' 'GitInviwo' 'GitGudhi']


In [4]:
# Which wiggle minimization is faster?
pivotwiggle = df.pivot_table(index=["file", "algorithm"], columns="weighted", values="twiggle")
print(pivotwiggle)

weighted                           False     True 
file                 algorithm                    
Cartilage            ILP             7.0      11.0
                     Median          9.0      11.0
Cylinder2D[500,1400] ILP         2,324.0   1,701.0
                     Median      1,953.0   1,541.0
Cylinder3D[50,150]   ILP            39.0      34.0
                     Median         43.0      47.0
DigestivePhysiology  ILP             8.0       8.0
                     Median          8.0       9.0
Europe               ILP           587.0     457.0
                     Median        457.0     487.0
GitGudhi             Median     68,679.0  36,615.0
GitPython            Median    524,864.0 141,041.0
Storm                ILP           619.0     690.0
                     Median        599.0     630.0
Tangora[40,199]      ILP            62.0      75.0
                     Median         84.0      69.0
UrinaryTract         ILP             9.0       6.0
                     Median    

In [7]:
pivotcrossings = df.pivot_table(index=["file", "algorithm"], columns="weighted", values="crossings")
print(pivotcrossings)

weighted                           False     True 
file                 algorithm                    
Cartilage            ILP             0.0       0.0
                     Median          0.0       0.0
Cylinder2D[500,1400] ILP             0.0       0.0
                     Median          2.0       2.0
Cylinder3D[50,150]   ILP             2.0       2.0
                     Median         13.0      13.0
DigestivePhysiology  ILP             0.0       0.0
                     Median          0.0       0.0
Europe               ILP             4.0       4.0
                     Median          4.0       4.0
GitGudhi             Median     10,214.0  10,214.0
GitPython            Median    462,031.0 462,031.0
Storm                ILP             7.0       7.0
                     Median         49.0      49.0
Tangora[40,199]      ILP             0.0       0.0
                     Median          3.0       3.0
UrinaryTract         ILP             0.0       0.0
                     Median    

In [13]:
# Create the latex table
import math
pivot = df.pivot_table(index = "file", columns = ["weighted", "algorithm"], values = ["twiggle", "tcross", "nodes", "hedges", "tedges", "timesteps", "levels", "crossings"])
# now create df for latex table
dfdict_latex = defaultdict(list)
for i, row in pivot.iterrows():
    dfdict_latex["Instance"].append(r"\textbf{" + i +r"}")
    dfdict_latex["$|V|$"].append(int(round(row[("nodes", False, "Median")])))
    dfdict_latex["$|E_H|$"].append(int(round(row[("hedges", False, "Median")])))
    dfdict_latex["$|E_T|$"].append(int(round(row[("tedges", False, "Median")])))
    dfdict_latex["$|T|$"].append(int(round(row[("timesteps", False, "Median")])))
    crilp = None
    if not math.isnan(row[("crossings", False, "ILP")]):
        dfdict_latex[r"$\text{cr}_{\text{ILP}}$"].append(int(round(row[("crossings", False, "ILP")])))
        crilp = int(round(row[("crossings", False, "ILP")]))
    elif not math.isnan(row[("crossings", True, "ILP")]):
        dfdict_latex[r"$\text{cr}_{\text{ILP}}$"].append(int(round(row[("crossings", True, "ILP")])))
    else:        
        dfdict_latex[r"$\text{cr}_{\text{ILP}}$"].append("-")
    dfdict_latex[r"$\text{cr}_{\text{Median}}$"].append(int(round((round(row[("crossings", False, "Median")])+round(row[("crossings", False, "Median")])) / 2)))
    ilpruntimes = []
    if not math.isnan(row[("tcross", False, "ILP")]):
        ilpruntimes.append(row[("tcross", False, "ILP")])
    if not math.isnan(row[("tcross", True, "ILP")]):
        ilpruntimes.append(row[("tcross", True, "ILP")])
    if len(ilpruntimes):
        dfdict_latex[r"$t_{\text{ILP}}$ [s]"].append(np.mean(ilpruntimes)/1000)
    else:
        dfdict_latex[r"$t_{\text{ILP}}$ [s]"].append("t.l.")
    dfdict_latex[r"$t_{\text{Median}}$ [s]"].append(np.mean([row[("tcross", False, "Median")], row[("tcross", True, "Median")]])/1000)

    wiggletimes = defaultdict(list)
    for alg in ["Median", "ILP"]:
        for weighted in [False, True]:
            if not math.isnan(row[("twiggle", weighted, alg)]):
                wiggletimes[weighted].append(row[("twiggle", weighted, alg)])    
    dfdict_latex[r"$t_{\text{wiggle}}$ [s]"].append(np.mean(wiggletimes[False])/1000)
    dfdict_latex[r"$t_{\text{wiggleW}}$ [s]"].append(np.mean(wiggletimes[True])/1000)
df_latex = pd.DataFrame(dfdict_latex)
df_latex.sort_values(by=["$|V|$"], inplace=True)
print(df_latex.to_latex(column_format="|l|r|r|r|r||r|r|r|r||r|r|", index=False, header=True, float_format="%.2f", bold_rows=True))

\begin{tabular}{|l|r|r|r|r||r|r|r|r||r|r|}
\toprule
Instance & $|V|$ & $|E_H|$ & $|E_T|$ & $|T|$ & $\text{cr}_{\text{ILP}}$ & $\text{cr}_{\text{Median}}$ & $t_{\text{ILP}}$ [s] & $t_{\text{Median}}$ [s] & $t_{\text{wiggle}}$ [s] & $t_{\text{wiggleW}}$ [s] \\
\midrule
\textbf{UrinaryTract} & 132 & 127 & 102 & 5 & 0 & 0 & 0.07 & 0.00 & 0.01 & 0.01 \\
\textbf{Cartilage} & 172 & 163 & 145 & 9 & 0 & 0 & 0.07 & 0.00 & 0.01 & 0.01 \\
\textbf{DigestivePhysiology} & 192 & 184 & 145 & 8 & 0 & 0 & 0.10 & 0.00 & 0.01 & 0.01 \\
\textbf{Cylinder3D[50,150]} & 896 & 795 & 742 & 101 & 2 & 13 & 0.05 & 0.01 & 0.04 & 0.04 \\
\textbf{ViscousFingers} & 978 & 918 & 943 & 60 & 10 & 83 & 0.43 & 0.01 & 0.08 & 0.07 \\
\textbf{Tangora[40,199]} & 1288 & 1137 & 1134 & 151 & 0 & 3 & 0.07 & 0.01 & 0.07 & 0.07 \\
\textbf{Europe} & 2565 & 2516 & 2525 & 49 & 4 & 4 & 19.73 & 0.03 & 0.52 & 0.47 \\
\textbf{Storm} & 8446 & 7702 & 7901 & 744 & 7 & 49 & 0.89 & 0.10 & 0.61 & 0.66 \\
\textbf{Cylinder2D[500,1400]} & 15224 & 1432